# **Sustainability Aware Asset Management :**
# **Portfolio Allocation with a Carbon Objective**

To reproduce the analysis, please follow the steps below :

- Clone the project repository locally.

- Create the environment: 

   conda env create -f environment.yml

- Activate the environment:

   conda activate saam-project

- Select the newly created kernel in the notebook.

- Run all cells sequentially, from top to bottom.

## **Part I - Standard Portfolio Allocation**
**The first part of the project consists of building a portfolio based on the minimum-variance criterion.**

In [ ]:
import pandas as pd
import numpy as np

#Define the path to the cleaned datasets
path = "../data/cleaned/"

#Load the cleaned datasets
ri_m = pd.read_csv(path + "RI_M_cleaned.csv", sep=";", na_values="N/A")
co2 = pd.read_csv(path + "CO2_S1_cleaned.csv", sep=";", na_values="N/A")
static = pd.read_csv(path + "STATIC_cleaned.csv", sep=";", na_values="N/A")

### **1) Data cleaning**

Data cleaning was necessary to design a meaningful investment strategy. 

The datasets used in this notebook are cleaned versions of the original Datastream files, based on the cleaning steps required by the project :

- group assignment: region and climate strategy (done in Excel)
- missing revenues and CO2 emissions (done in Excel)
- missing prices (done in Excel)
- missing values (done in Excel)
- low prices (done in Excel)

The table below shows how many firms remain after these initial cleaning steps :

In [ ]:
print("static shape:", static.shape)
pd.concat([static.head(), static.tail()])

- stale prices (addressed later in Section 2.1)
- insufficient return history (addressed later in Section 2.1)
- carbon data availability (addressed later in Section 2.1)
- allocation time horizon (addressed later in Section 2.1)

A more detailed description of the cleaning procedure is provided in the report.

We check that the cleaned datasets contain the same number of firms and then organise the matrices into a more suitable format :

We keep 2025 in the CO2 dataset for carry-forward purposes, but not to construct portfolio weights for 2026 ...

In [ ]:
print("co2 shape:", co2.shape)
co2.head()

In [ ]:
#Read the year labels as integers
years = pd.to_numeric(co2.columns[2:], errors="raise").astype(int)

#Create a copy of co2 and assign the year labels to the columns
co2_wide = co2.copy()
co2_wide.columns = ["NAME", "ISIN"] + list(years)

#Use ISIN and NAME as row identifiers and sort the columns by year
co2_wide = co2_wide.set_index(["ISIN", "NAME"]).sort_index(axis=1)

#Transform the CO2 values into numeric format
co2_wide = co2_wide.apply(
    lambda col: pd.to_numeric(
        col.astype(str).str.replace(",", ".", regex=False),
        errors="coerce"
    )
)

print("co2_wide shape:", co2_wide.shape)
pd.concat([co2_wide.head(), co2_wide.tail()])

... since the full set of monthly returns for 2026 is not available, as shown below.

In [ ]:
print("ri_m shape:", ri_m.shape)
ri_m.head()

In [ ]:
#Convert the column labels into dates
dates = pd.to_datetime(ri_m.columns[2:], format="%d/%m/%y", errors="coerce")

#Create a copy of ri_m and assign the date labels to the columns
ri_m_wide = ri_m.copy()
ri_m_wide.columns = ["NAME", "ISIN"] + list(dates)

#Use ISIN and NAME as row identifiers and sort the columns by date
ri_m_wide = ri_m_wide.set_index(["ISIN", "NAME"]).sort_index(axis=1)

#Transform the return index values into numeric format
ri_m_wide = ri_m_wide.apply(
    lambda col: pd.to_numeric(
        col.astype(str).str.replace(",", ".", regex=False),
        errors="coerce"
    )
)

print("ri_m_wide shape:", ri_m_wide.shape)
pd.concat([ri_m_wide.head(), ri_m_wide.tail()])

### **2) Minimum-Variance Portfolio Allocation**

The return index values are converted into returns using the simple return definition, $R_{i,t} = \frac{P_{i,t}}{P_{i,t-1}} - 1$.

Returns at date *t* are then collected in the vector $R_t = (R_{1,t}, \ldots, R_{N,t})$, where $N$ denotes the number of firms.

In [ ]:
#Convert return index values into simple returns by computing percentage changes over time
returns_wide = ri_m_wide.pct_change(axis=1, fill_method=None)

print("returns_wide shape:", returns_wide.shape)
pd.concat([returns_wide.head(), returns_wide.tail()])

#### **2.1) Investment set** 

This section defines the investment sets used to construct **out of sample** portfolios.

The allocation exercise is carried out from **the end of 2013 to the end of 2024**.

**At the end of each year $Y$**, the investment set is defined as the set of firms that satisfy the following criteria :

- sufficient return history to compute expected returns and the covariance matrix (at least 3 years of available monthly returns over the past 10 years)
- no stale prices (the share of zero returns, computed over available monthly returns, must not exceed 50%)
- availability of carbon data

If a firm does not satisfy these criteria at the end of year $Y$, it is excluded from that year’s investment set.

In [ ]:
#Years used to determine portfolio weights (the last formation year is 2024, not 2025)
years = range(2013, 2025)

#Filtering parameters
tau = 120   #10 years of monthly returns
min_obs = 36   #at least 3 years of available monthly returns
stale_threshold = 0.50   #50% of zero returns among available monthly returns

#Store the yearly investment sets and the corresponding estimation inputs
investment_sets = {}   #firms in the investment set
mu_dict = {}   #expected return vector
sigma_dict = {}   #covariance matrix    

for Y in years:

    #Select the 10-year monthly return window up to December of year Y
    end_date = pd.Timestamp(f"{Y}-12-31")
    window_cols = returns_wide.columns[returns_wide.columns <= end_date][-tau:]
    returns_window = returns_wide[window_cols]

    #Filter 1: firms with sufficient return history
    n_available = returns_window.notna().sum(axis=1)
    valid_obs = n_available >= min_obs

    #Filter 2: firms not subject to stale prices
    zero_counts = returns_window.eq(0).sum(axis=1)
    zero_share = zero_counts / n_available   #the share of zero returns is computed over available monthly returns only
    not_stale = zero_share <= stale_threshold

    #Filter 3: firms with carbon data available at the end of year Y
    carbon_available = co2_wide[Y].notna()

    #Combine all filters
    eligible_firms = valid_obs & not_stale & carbon_available

    #Construct the final investment set for year Y
    investment_set = returns_window.loc[eligible_firms]
    investment_sets[Y] = investment_set.index

    #Compute expected returns and covariance matrix
    mu_dict[Y] = investment_set.mean(axis=1)
    sigma_dict[Y] = investment_set.T.cov(ddof=1)

The table below shows the size of the investment set over time, together with the dimensions of the expected return vector and covariance matrix :

In [ ]:
summary_table = pd.DataFrame({
    "Number of firms": {Y: len(investment_sets[Y]) for Y in years},
    "Expected return vector": {Y: mu_dict[Y].shape for Y in years},
    "Covariance matrix": {Y: sigma_dict[Y].shape for Y in years},
})

summary_table

As time passes, an increasing number of firms are included in the investment set.

The outputs below show the vectors of expected returns for 2013 and 2024, the first and last allocation years : 

In [ ]:
mu_dict[2013].head(477)

In [ ]:
mu_dict[2024].head(613)

The outputs below show the covariance matrices for the first five firms in 2013 and 2024, the first and last allocation years :

In [ ]:
#Modify the slicing to display the full covariance matrix
sigma_dict[2013].iloc[:5, :5]

In [ ]:
# Modify the slicing to display the full covariance matrix
sigma_dict[2024].iloc[:5, :5]

#### **2.2) Minimum-variance portfolio allocation**

In [ ]:
from __future__ import annotations

import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.optimize import minimize

# =========================================================
# SETTINGS
# =========================================================
# Notebook-friendly paths:
# run this from the notebooks folder, so ../data/cleaned exists
DATA_DIR = Path("../data/cleaned")
RI_FILE = DATA_DIR / "RI_M_cleaned.csv"
CO2_FILE = DATA_DIR / "CO2_S1_cleaned.csv"
RF_FILE = DATA_DIR / "RF.csv"

OUTPUT_DIR = Path("../results/outputs_minvar")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

START_FORMATION_YEAR = 2013   # first portfolio formed at end-2013
END_FORMATION_YEAR = 2024     # last portfolio formed at end-2024
MIN_OBS_WINDOW = 36           # at least 3 years of monthly returns in rolling 10y window
STALE_ZERO_THRESHOLD = 0.50   # exclude if more than 50% of returns are zero
RIDGE = 1e-8                  # numerical stabilization for covariance matrix
SOLVER_MAXITER = 2000
FTOL = 1e-12
CSV_SEP = ";"

warnings.filterwarnings("ignore", category=RuntimeWarning)

# =========================================================
# HELPERS
# =========================================================
def to_numeric_panel(df: pd.DataFrame) -> pd.DataFrame:
    """Convert object cells using comma decimal if present."""
    return df.apply(
        lambda col: pd.to_numeric(
            col.astype(str).str.replace(",", ".", regex=False),
            errors="coerce"
        )
    )


def parse_calendar_date_headers(columns: list[str]) -> list[pd.Timestamp]:
    """
    Convert string date headers like 31/12/1999 into pandas Timestamps.
    We parse one by one to avoid pandas format-inference warnings.
    """
    cols = pd.Index(columns).astype(str).str.strip()
    return [pd.to_datetime(c, dayfirst=True) for c in cols]


def parse_rf_dates(date_series: pd.Series) -> pd.DatetimeIndex:
    """
    Parse RF dates robustly:
    - if they look numeric, treat them as Excel serial dates
    - otherwise parse them as day-first calendar dates
    """
    raw = date_series.astype(str).str.strip()
    numeric = pd.to_numeric(raw, errors="coerce")

    if numeric.notna().all():
        return pd.DatetimeIndex(pd.to_datetime(numeric, unit="D", origin="1899-12-30"))

    return pd.DatetimeIndex(pd.to_datetime(raw, dayfirst=True, errors="coerce"))


def annualized_return(monthly_returns: pd.Series) -> float:
    r = monthly_returns.dropna()
    if len(r) == 0:
        return np.nan
    compounded = (1.0 + r).prod()
    return compounded ** (12.0 / len(r)) - 1.0


def annualized_volatility(monthly_returns: pd.Series) -> float:
    r = monthly_returns.dropna()
    if len(r) < 2:
        return np.nan
    return r.std(ddof=1) * np.sqrt(12.0)


def max_drawdown(cumulative_index: pd.Series) -> float:
    running_max = cumulative_index.cummax()
    drawdown = cumulative_index / running_max - 1.0
    return drawdown.min()


# =========================================================
# LOAD CLEANED CSV FILES
# =========================================================
print("Loading cleaned CSV files...")

ri_raw = pd.read_csv(RI_FILE, sep=CSV_SEP, na_values=["N/A", "#N/A", "nan", "NaN", ""])
co2_raw = pd.read_csv(CO2_FILE, sep=CSV_SEP, na_values=["N/A", "#N/A", "nan", "NaN", ""])
rf_raw = pd.read_csv(RF_FILE, sep=CSV_SEP, na_values=["N/A", "#N/A", "nan", "NaN", ""])

# =========================================================
# PREPARE MONTHLY RI PANEL
# =========================================================
print("Preparing monthly RI panel...")

# NOTE: in the new cleaned RI_M file, date columns are already calendar dates,
# not Excel serial numbers.
ri_date_cols = parse_calendar_date_headers(list(ri_raw.columns[2:]))

ri = ri_raw.copy()
ri.columns = ["NAME", "ISIN"] + ri_date_cols
ri = ri.set_index(["ISIN", "NAME"])
ri = to_numeric_panel(ri)

# NOTE:
# We do NOT mask RI < 0.5 here, because the cleaned CSV is assumed to
# already reflect your chosen data-cleaning rules, and explicit delisting
# zeros must be preserved.

# Official project output ends at Dec-2025
ri = ri.loc[:, ri.columns <= pd.Timestamp("2025-12-31")]

print("RI panel shape:", ri.shape)

# =========================================================
# CONSTRUCT MONTHLY RETURNS
# =========================================================
print("Constructing monthly simple returns...")

# IMPORTANT:
# fill_method=None preserves missing values and keeps the delisting treatment
# already embedded in the cleaned RI file.
returns = ri.pct_change(axis=1, fill_method=None)

print("Returns panel shape:", returns.shape)

# =========================================================
# PREPARE ANNUAL CO2 PANEL
# =========================================================
print("Preparing annual CO2 panel...")

co2_year_cols = list(pd.to_numeric(co2_raw.columns[2:], errors="raise").astype(int))
co2 = co2_raw.copy()
co2.columns = ["NAME", "ISIN"] + co2_year_cols
co2 = co2.set_index(["ISIN", "NAME"])
co2 = to_numeric_panel(co2)

# NOTE:
# We do NOT apply carry-forward here, because the cleaned annual CO2 file
# is assumed to already contain the cleaning choices you made.

print("CO2 panel shape:", co2.shape)

# Common index across RI and CO2
common_index = returns.index.intersection(co2.index)
ri = ri.loc[common_index]
returns = returns.loc[common_index]
co2 = co2.loc[common_index]

# =========================================================
# PREPARE RISK-FREE RATE
# =========================================================
print("Preparing risk-free rate...")

rf = rf_raw.copy()
if rf.shape[1] < 2:
    raise ValueError("RF.csv must contain at least one date column and one RF column.")

rf_date_col = rf.columns[0]
rf_value_col = rf.columns[1]

rf_dates = parse_rf_dates(rf[rf_date_col])
rf_values = pd.to_numeric(
    rf[rf_value_col].astype(str).str.replace(",", ".", regex=False),
    errors="coerce"
)

rf_series = pd.Series(rf_values.to_numpy(), index=rf_dates).sort_index()
rf_series = rf_series[~rf_series.index.isna()]
rf_series = rf_series.loc[rf_series.index <= pd.Timestamp("2025-12-31")]

# If RF looks like percentages > 1, scale down to decimals
rf_nonmissing = rf_series.dropna()
if len(rf_nonmissing) > 0 and rf_nonmissing.median() > 1:
    rf_series = rf_series / 100.0

print("RF series length:", len(rf_series))

# =========================================================
# BUILD YEAR-END MAP
# =========================================================
print("Building year-end map...")

december_dates = [d for d in ri.columns if d.month == 12]
year_end_dec = {}

for y in sorted({d.year for d in december_dates}):
    year_dates = [d for d in december_dates if d.year == y]
    if year_dates:
        year_end_dec[y] = max(year_dates)

required_years = range(START_FORMATION_YEAR, END_FORMATION_YEAR + 1)
missing_years = [y for y in required_years if y not in year_end_dec]
if missing_years:
    raise ValueError(f"Missing December RI dates for years: {missing_years}")

# =========================================================
# INVESTMENT-SET FUNCTION
# =========================================================
def get_investment_set(formation_year: int) -> pd.MultiIndex:
    dec_date = year_end_dec[formation_year]
    window_start = dec_date - pd.DateOffset(months=119)

    window_returns = returns.loc[
        :,
        (returns.columns >= window_start) & (returns.columns <= dec_date)
    ]

    # Rule 1: carbon available at end of year Y
    if formation_year in co2.columns:
        carbon_available = co2[formation_year].notna()
    else:
        carbon_available = pd.Series(False, index=co2.index)

    # Rule 2: valid December RI at end of year Y
    # (if price is missing at end of Y, do not invest in Y+1)
    valid_december_ri = ri[dec_date].notna()

    # Rule 3: enough monthly return history
    valid_return_counts = window_returns.notna().sum(axis=1)
    sufficient_history = valid_return_counts >= MIN_OBS_WINDOW

    # Rule 4: stale-price filter
    # Following the literal 10-year window interpretation: denominator = 120
    zero_counts = window_returns.eq(0).sum(axis=1)
    zero_prop = zero_counts / 120.0
    not_stale = zero_prop <= STALE_ZERO_THRESHOLD

    eligible_mask = carbon_available & valid_december_ri & sufficient_history & not_stale
    return common_index[eligible_mask.loc[common_index].to_numpy()]

# =========================================================
# MIN-VARIANCE SOLVER
# =========================================================
def solve_minvar(cov_matrix: pd.DataFrame) -> pd.Series:
    n = cov_matrix.shape[0]
    if n == 0:
        raise ValueError("Covariance matrix is empty.")
    if n == 1:
        return pd.Series([1.0], index=cov_matrix.index)

    Sigma = cov_matrix.to_numpy(dtype=float)
    Sigma = Sigma + RIDGE * np.eye(n)
    x0 = np.repeat(1.0 / n, n)

    def objective(w: np.ndarray) -> float:
        return float(w @ Sigma @ w)

    constraints = [{"type": "eq", "fun": lambda w: np.sum(w) - 1.0}]
    bounds = [(0.0, 1.0)] * n

    res = minimize(
        objective,
        x0=x0,
        method="SLSQP",
        bounds=bounds,
        constraints=constraints,
        options={"maxiter": SOLVER_MAXITER, "ftol": FTOL, "disp": False},
    )

    if not res.success:
        raise ValueError(f"Optimization failed: {res.message}")

    weights = pd.Series(res.x, index=cov_matrix.index)
    weights = weights / weights.sum()
    return weights

# =========================================================
# MAIN YEARLY LOOP
# =========================================================
print("Running yearly minimum-variance loop...")

investment_set_rows = []
weights_long_rows = []
oos_rows = []

for Y in range(START_FORMATION_YEAR, END_FORMATION_YEAR + 1):
    print(f"  Formation year {Y} -> OOS year {Y + 1}")

    eligible = get_investment_set(Y)
    dec_date = year_end_dec[Y]
    window_start = dec_date - pd.DateOffset(months=119)

    est_window = returns.loc[
        eligible,
        (returns.columns >= window_start) & (returns.columns <= dec_date)
    ]

    investment_set_rows.append({"formation_year": Y, "n_eligible": len(eligible)})

    if len(eligible) == 0:
        print(f"    WARNING: no eligible firms for year {Y}")
        continue

    # Practical covariance stability filter
    est_window = est_window.loc[est_window.notna().sum(axis=1) >= 2]
    if est_window.empty:
        print(f"    WARNING: no firms remain after covariance stability filter for year {Y}")
        continue

    # Fill missing estimation-window returns with each asset's own mean
    # (practical choice for covariance estimation)
    asset_means = est_window.mean(axis=1)
    est_window_filled = est_window.T.fillna(asset_means).T

    Sigma = est_window_filled.T.cov()

    # Drop any residual problematic rows/cols
    valid_assets = Sigma.index[~Sigma.isna().all(axis=1)]
    Sigma = Sigma.loc[valid_assets, valid_assets]
    Sigma = Sigma.dropna(axis=0, how="any").dropna(axis=1, how="any")

    if Sigma.empty:
        print(f"    WARNING: covariance matrix empty after cleanup for year {Y}")
        continue

    w0 = solve_minvar(Sigma)

    for (isin, name), w in w0.items():
        weights_long_rows.append({
            "formation_year": Y,
            "oos_year": Y + 1,
            "ISIN": isin,
            "NAME": name,
            "weight": w,
        })

    # Out-of-sample months in Y+1
    oos_months = sorted([d for d in returns.columns if d.year == Y + 1])
    if len(oos_months) == 0:
        continue

    w = w0.copy()

    for dt in oos_months:
        r_t = returns.loc[w.index, dt]

        # -------------------------------------------------
        # IMPORTANT:
        # This line is conservative and may need discussion.
        # It treats unexpected missing post-formation returns
        # as -100% losses.
        # If you do NOT want this convention, this is the main
        # line to revisit later.
        # -------------------------------------------------
        r_t = r_t.fillna(-1.0)

        rp_t = float((w * r_t).sum())

        oos_rows.append({
            "date": dt,
            "formation_year": Y,
            "oos_year": Y + 1,
            "portfolio_return": rp_t,
        })

        # Update weights passively within the year
        denom = 1.0 + rp_t
        if abs(denom) > 1e-12:
            w = w * (1.0 + r_t) / denom
            total = w.sum()
            if abs(total) > 1e-12:
                w = w / total
            else:
                w = pd.Series(np.repeat(1.0 / len(w), len(w)), index=w.index)
        else:
            w = pd.Series(np.repeat(1.0 / len(w), len(w)), index=w.index)

# =========================================================
# OUTPUT TABLES
# =========================================================
print("Saving outputs...")

investment_set_sizes = pd.DataFrame(investment_set_rows)
weights_by_year_long = pd.DataFrame(weights_long_rows)
minvar_monthly_returns = pd.DataFrame(oos_rows).sort_values("date").reset_index(drop=True)

if minvar_monthly_returns.empty:
    raise ValueError("No out-of-sample minimum-variance returns were produced.")

minvar_monthly_returns["date"] = pd.to_datetime(minvar_monthly_returns["date"])
minvar_monthly_returns = minvar_monthly_returns[
    (minvar_monthly_returns["date"] >= pd.Timestamp("2014-01-01")) &
    (minvar_monthly_returns["date"] <= pd.Timestamp("2025-12-31"))
].copy()

minvar_monthly_returns["cumulative_index"] = (1.0 + minvar_monthly_returns["portfolio_return"]).cumprod()
minvar_cumulative_returns = minvar_monthly_returns[
    ["date", "formation_year", "oos_year", "portfolio_return", "cumulative_index"]
].copy()

# Template-ready table: one row per month from 2014-01 to 2025-12
full_months = pd.date_range("2014-01-31", "2025-12-31", freq="ME")
template_ready = pd.DataFrame({"date": full_months})
template_ready = template_ready.merge(
    minvar_monthly_returns[["date", "portfolio_return"]],
    on="date",
    how="left"
)
template_ready["date_str"] = template_ready["date"].dt.strftime("%Y-%m")
template_ready = template_ready[["date_str", "portfolio_return"]]

# Summary stats
r = minvar_monthly_returns.set_index("date")["portfolio_return"].sort_index()
rf_monthly = rf_series.reindex(r.index)

if rf_monthly.isna().all():
    sharpe = np.nan
else:
    rf_monthly = rf_monthly.ffill().bfill()
    excess = r - rf_monthly
    ex_ann = annualized_return(excess)
    vol_ann = annualized_volatility(r)
    sharpe = ex_ann / vol_ann if pd.notna(ex_ann) and pd.notna(vol_ann) and vol_ann != 0 else np.nan

summary_stats = pd.DataFrame({
    "metric": [
        "Annualized average return",
        "Annualized volatility",
        "Sharpe ratio",
        "Minimum monthly return",
        "Maximum monthly return",
        "Final cumulative index",
        "Maximum drawdown",
        "Number of monthly observations",
    ],
    "value": [
        annualized_return(r),
        annualized_volatility(r),
        sharpe,
        r.min(),
        r.max(),
        minvar_monthly_returns["cumulative_index"].iloc[-1],
        max_drawdown(minvar_monthly_returns.set_index("date")["cumulative_index"]),
        int(r.notna().sum()),
    ]
})

# Save files
investment_set_sizes.to_csv(OUTPUT_DIR / "investment_set_sizes.csv", index=False)
weights_by_year_long.to_csv(OUTPUT_DIR / "minvar_weights_by_year_long.csv", index=False)
minvar_monthly_returns.to_csv(OUTPUT_DIR / "minvar_monthly_returns.csv", index=False)
minvar_cumulative_returns.to_csv(OUTPUT_DIR / "minvar_cumulative_returns.csv", index=False)
summary_stats.to_csv(OUTPUT_DIR / "minvar_summary_stats.csv", index=False)
template_ready.to_csv(OUTPUT_DIR / "minvar_template_ready.csv", index=False)

print("Done.")
print(f"Outputs saved in: {OUTPUT_DIR}")
print(investment_set_sizes.to_string(index=False))
print("\nSummary stats:")
print(summary_stats.to_string(index=False))

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

BASE_DIR = Path.cwd()

# If the notebook is running from /notebooks, move one level up
PROJECT_DIR = BASE_DIR.parent if BASE_DIR.name == "notebooks" else BASE_DIR

# Try the most likely output locations
if (PROJECT_DIR / "outputs_minvar").exists():
    OUT_DIR = PROJECT_DIR / "outputs_minvar"
elif (PROJECT_DIR / "results" / "outputs_minvar").exists():
    OUT_DIR = PROJECT_DIR / "results" / "outputs_minvar"
else:
    raise FileNotFoundError(
        "Could not find 'outputs_minvar'. Checked both:\n"
        f"- {PROJECT_DIR / 'outputs_minvar'}\n"
        f"- {PROJECT_DIR / 'results' / 'outputs_minvar'}"
    )

# ----------------------------
# Load files
# ----------------------------
investment = pd.read_csv(OUT_DIR / "investment_set_sizes.csv")
monthly = pd.read_csv(OUT_DIR / "minvar_monthly_returns.csv")
cumulative = pd.read_csv(OUT_DIR / "minvar_cumulative_returns.csv")
weights = pd.read_csv(OUT_DIR / "minvar_weights_by_year_long.csv")
summary = pd.read_csv(OUT_DIR / "minvar_summary_stats.csv")

monthly["date"] = pd.to_datetime(monthly["date"])
cumulative["date"] = pd.to_datetime(cumulative["date"])

# ----------------------------
# Graph 1: Investment set size by formation year
# ----------------------------
plt.figure(figsize=(10, 6))
plt.plot(investment["formation_year"], investment["n_eligible"], marker="o")
plt.title("Eligible Universe Size by Formation Year")
plt.xlabel("Formation Year")
plt.ylabel("Number of Eligible Firms")
plt.grid(True)
plt.tight_layout()
plt.savefig(OUT_DIR / "graph_investment_set_sizes.png", dpi=300)
plt.close()

# ----------------------------
# Graph 2: Cumulative performance
# ----------------------------
plt.figure(figsize=(12, 6))
plt.plot(cumulative["date"], cumulative["cumulative_index"])
plt.title("Minimum-Variance Portfolio Cumulative Performance")
plt.xlabel("Date")
plt.ylabel("Cumulative Index")
plt.grid(True)
plt.tight_layout()
plt.savefig(OUT_DIR / "graph_cumulative_performance.png", dpi=300)
plt.close()

# ----------------------------
# Graph 3: Monthly returns through time
# ----------------------------
plt.figure(figsize=(12, 6))
plt.bar(monthly["date"], monthly["portfolio_return"])
plt.title("Minimum-Variance Portfolio Monthly Returns")
plt.xlabel("Date")
plt.ylabel("Monthly Return")
plt.grid(True)
plt.tight_layout()
plt.savefig(OUT_DIR / "graph_monthly_returns.png", dpi=300)
plt.close()

# ----------------------------
# Graph 4: Annual returns bar chart
# ----------------------------
monthly["year"] = monthly["date"].dt.year
annual_returns = (
    monthly.groupby("year")["portfolio_return"]
    .apply(lambda x: (1 + x).prod() - 1)
    .reset_index()
)

plt.figure(figsize=(10, 6))
plt.bar(annual_returns["year"].astype(str), annual_returns["portfolio_return"])
plt.title("Minimum-Variance Portfolio Annual Returns")
plt.xlabel("Year")
plt.ylabel("Annual Return")
plt.grid(True)
plt.tight_layout()
plt.savefig(OUT_DIR / "graph_annual_returns.png", dpi=300)
plt.close()

# ----------------------------
# Graph 5: Top 10 weights in latest formation year
# ----------------------------
latest_year = int(weights["formation_year"].max())
top_weights = (
    weights[weights["formation_year"] == latest_year]
    .sort_values("weight", ascending=False)
    .head(10)
    .sort_values("weight", ascending=True)
)

plt.figure(figsize=(10, 6))
plt.barh(top_weights["NAME"], top_weights["weight"])
plt.title(f"Top 10 Portfolio Weights - Formation Year {latest_year}")
plt.xlabel("Weight")
plt.ylabel("Company")
plt.tight_layout()
plt.savefig(OUT_DIR / "graph_top10_weights_latest_year.png", dpi=300)
plt.close()

# ----------------------------
# Graph 6: Rolling 12-month return
# ----------------------------
rolling_12m = monthly[["date", "portfolio_return"]].copy()
rolling_12m["rolling_12m_return"] = (
    (1 + rolling_12m["portfolio_return"])
    .rolling(12)
    .apply(lambda x: x.prod() - 1, raw=True)
)

plt.figure(figsize=(12, 6))
plt.plot(rolling_12m["date"], rolling_12m["rolling_12m_return"])
plt.title("Rolling 12-Month Return")
plt.xlabel("Date")
plt.ylabel("Rolling 12-Month Return")
plt.grid(True)
plt.tight_layout()
plt.savefig(OUT_DIR / "graph_rolling_12m_return.png", dpi=300)
plt.close()

print("Graphs created successfully.")
print()
print("Files created:")
print("- graph_investment_set_sizes.png")
print("- graph_cumulative_performance.png")
print("- graph_monthly_returns.png")
print("- graph_annual_returns.png")
print("- graph_top10_weights_latest_year.png")
print("- graph_rolling_12m_return.png")

## **Part II - Portfolio Allocation with Carbon Emission Reduction**
**The second part of the project focuses on the climate impact of the portfolio.**
